<a href="https://colab.research.google.com/github/yourblandcat/mlLab/blob/main/ml_lab_multipleRegressionBostonHouseDataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Regression Models
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# 1. Load the Dataset
data_url = "http://lib.stat.cmu.edu/datasets/boston"
raw_df = pd.read_csv(data_url, sep=r"\s+", skiprows=22, header=None)

# Format features and target (MEDV)
data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
target = raw_df.values[1::2, 2]

feature_names = [
    'CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM',
    'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT'
]

X = pd.DataFrame(data, columns=feature_names)
y = pd.Series(target, name='MEDV')

# 2. Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. Feature Scaling (essential for linear models)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Define Models to Compare
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
}

# 5. Train & Evaluate Models
results = []

for name, model in models.items():
    # Tree models don't strictly require scaling, but linear models do
    if "Regression" in name:
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

    # Calculate performance metrics
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    results.append({
        "Model": name,
        "RMSE ($1k)": round(rmse, 3),
        "MAE ($1k)": round(mae, 3),
        "R² Score": round(r2, 4)
    })

# Display Results
comparison_df = pd.DataFrame(results).sort_values(by="R² Score", ascending=False)
print(comparison_df.to_string(index=False))

            Model  RMSE ($1k)  MAE ($1k)  R² Score
Gradient Boosting       2.492      1.912    0.9153
    Random Forest       2.811      2.040    0.8923
Linear Regression       4.929      3.189    0.6688
 Ridge Regression       4.931      3.186    0.6685


In [2]:
import pandas as pd

# 1. Define custom property features matching the exact columns:
# ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT']
my_house = pd.DataFrame([{
    'CRIM': 0.03,     # Per capita crime rate
    'ZN': 0.0,        # Proportion of residential land zoned for large lots
    'INDUS': 7.07,    # Proportion of non-retail business acres
    'CHAS': 0,        # Charles River dummy variable (1 if tract bounds river; 0 otherwise)
    'NOX': 0.469,     # Nitric oxides concentration
    'RM': 6.5,        # Average number of rooms per dwelling
    'AGE': 65.2,      # Proportion of owner-occupied units built prior to 1940
    'DIS': 4.4,       # Weighted distances to Boston employment centers
    'RAD': 24,        # Index of accessibility to radial highways
    'TAX': 242,       # Full-value property-tax rate per $10,000
    'PTRATIO': 17.8,  # Pupil-teacher ratio by town
    'B': 396.9,       # Proportion of Black residents
    'LSTAT': 9.14     # % lower status of the population
}])

# 2. Extract best model (e.g., Gradient Boosting model trained earlier)
best_model = models["Gradient Boosting"]

# 3. Predict!
predicted_price_thousands = best_model.predict(my_house)

print(f"Estimated House Value: ${predicted_price_thousands[0] * 1000:,.2f}")

Estimated House Value: $21,817.96


In [5]:
# custom prediction

from sklearn.impute import SimpleImputer

# Create an imputer that fills missing fields with training set averages
imputer = SimpleImputer(strategy='mean')
imputer.fit(X_train)

# User provides partial data (e.g., missing NOX, TAX, PTRATIO)
partial_house = pd.DataFrame([{
    'CRIM': 0.03,
    'RM': 6.5,
    'AGE': 65.2,
    'LSTAT': 9.14,
    'CHAS': 0
}])

# Ensure all columns exist, filling missing ones with NaN
for col in X_train.columns:
    if col not in partial_house.columns:
        partial_house[col] = np.nan

# Reorder columns to match exact training format
partial_house = partial_house[X_train.columns]

# Fill missing NaNs with training averages
completed_house = imputer.transform(partial_house)

# Predict normally!
predicted_price = best_model.predict(completed_house)

print(f"Estimated House Value: ${predicted_price[0] * 1000:,.2f}")

Estimated House Value: $22,986.51


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
